# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to flattened non-overlapping patch tokens.

    Args:
        x: Image batch with shape (batch_size, channels, height, width).
        patch_size: Side length of each square patch.

    Returns:
        Tensor with shape (batch_size, num_patches, channels * patch_size * patch_size).
    """
    batch_size, channels, height, width = x.shape

    if height % patch_size != 0 or width % patch_size != 0:
        raise ValueError("Image height and width must be divisible by patch_size")

    patches_h = height // patch_size
    patches_w = width // patch_size

    x = x.reshape(batch_size, channels, patches_h, patch_size, patches_w, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5).contiguous() #permute() happens in the tensor view, contiguous is for the memory()
    return x.reshape(batch_size, patches_h * patches_w, channels * patch_size * patch_size)

In [3]:
# Patch projection and learned positional encoding from ViT.
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int): #linear projection 
        super().__init__()
        self.proj = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        """Project flattened patches from patch_dim to d_model."""
        return self.proj(x_patches)


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens + 1, d_model))

        #fill this tensor with values from a normal distribution, but truncate extreme values.
        #std=0.02 means the random values are very small, close to zero
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Prepend the class token and add learned positional embeddings."""
        batch_size = x.shape[0]
        cls_tokens = self.cls_token.expand(batch_size, -1, -1) #-1 means same dimension
        #concatenate with the class token x.shape = (batch_size, num_patches + 1, d_model)
        x = torch.cat([cls_tokens, x], dim=1)
        return x + self.pos_embed[:, : x.shape[1], :] #position vectors are added to patches in the batch

In [4]:
# FFN baseline and GLU variants to compare in the Transformer blocks.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout), #dropping out some percent of neurons during the training phase
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GLUFeedForward(nn.Module):
    """GLU-family FFN: act(W_a x) * W_b x, followed by projection back to d_model."""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        self.variant = variant.lower()
        self.proj_in = nn.Linear(d_model, 2 * d_ff_gated) #self.proj_in(x).shape == (batch_size, tokens, 512)
        self.dropout = nn.Dropout(dropout)
        self.proj_out = nn.Linear(d_ff_gated, d_model)

        valid_variants = {"geglu", "swiglu", "reglu", "glu"}
        if self.variant not in valid_variants:
            raise ValueError(f"Unknown GLU variant '{variant}'. Choose one of {sorted(valid_variants)}.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a, gate = self.proj_in(x).chunk(2, dim=-1) #split the tensor into two parts. take the last batch to GLU
        #what GLU does activation(a) * gate

#activation functions
        if self.variant == "geglu":
            a = F.gelu(a)
        elif self.variant == "swiglu":
            a = F.silu(a)
        elif self.variant == "reglu":
            a = F.relu(a)
        elif self.variant == "glu":
            a = torch.sigmoid(a)
        #proj_in -> split -> activation/gate multiply -> Dropout -> proj_out -> Dropout
        x = a * gate
        x = self.dropout(x) #drop hidden features before output projection
        x = self.proj_out(x)
        return self.dropout(x) #final out put of the MLP before residual addition

In [5]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block: #LN means LayerNorm, normalizing across the feature dimension 
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    #basically following the architecture of the paper 
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention( #nn.MultiheadAttention expects query, key and value
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = mlp

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_input = self.norm1(x)
        attn_out, _ = self.attn(attn_input, attn_input, attn_input, need_weights=False) #for query, key and value
        x = x + self.dropout(attn_out)
        x = x + self.mlp(self.norm2(x))
        return x

In [6]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> class token -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid #means patches_h and patches_w since MNIST is square 
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        self.patch_embed = PatchEmbed(patch_dim=patch_dim, d_model=d_model)
        self.pos_embed = PositionalEmbedding(num_tokens=self.num_tokens, d_model=d_model)
        self.dropout = nn.Dropout(dropout)

        def make_mlp() -> nn.Module:
            kind = mlp_kind.lower()
            if kind in {"ffn", "gelu", "baseline"}:
                return FeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)
            return GLUFeedForward(
                d_model=d_model,
                d_ff_gated=int(2 * d_ff / 3),
                dropout=dropout,
                variant=kind,
            )

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=make_mlp(),
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 10) #MNIST is classifying 0 to 9

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = patchify(x, self.patch_size)
        x = self.patch_embed(x)
        x = self.pos_embed(x)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        cls_token = self.norm(x[:, 0]) #similar to x[:, 0, :]
        logits = self.head(cls_token)  #passed to final classifier head
        return logits

In [7]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [9]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    epoch_train_losses: list[float] = []
    test_accs: list[float] = []
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad) #number of parameters need for training

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        running_loss = 0.0
        num_batches = 0
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())
            running_loss += loss.item()
            num_batches += 1

        epoch_train_losses.append(running_loss / num_batches)

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(
            f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | "
            f"train loss: {epoch_train_losses[-1]:.4f} | "
            f"test acc: {test_accs[-1]:.4f}"
        )

    return {
        "mlp_kind": mlp_kind,
        "num_params": num_params,
        "train_losses": train_losses,
        "epoch_train_losses": epoch_train_losses,
        "test_accs": test_accs,
        "final_train_loss": epoch_train_losses[-1],
        "final_test_acc": test_accs[-1],
        "best_test_acc": max(test_accs),
    }

In [11]:
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="cpu")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model example. TODO: You're welcome to experiment with these parameters
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

runs = ["baseline", "geglu", "swiglu"]
results = []

for kind in runs:
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nRun: {kind} | params: {num_params:,}")
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    results.append(out)

for result in results:
    print(
        f"{result['mlp_kind']}: final acc={result['final_test_acc']:.4f}, "
        f"best acc={result['best_test_acc']:.4f}, params={result['num_params']:,}"
    )  


Run: baseline | params: 105,098
[baseline] epoch 1/5 | train loss: 1.3564 | test acc: 0.8165
[baseline] epoch 2/5 | train loss: 0.5423 | test acc: 0.9031
[baseline] epoch 3/5 | train loss: 0.3814 | test acc: 0.9239
[baseline] epoch 4/5 | train loss: 0.3105 | test acc: 0.9398
[baseline] epoch 5/5 | train loss: 0.2707 | test acc: 0.9462

Run: geglu | params: 105,010
[geglu] epoch 1/5 | train loss: 1.2397 | test acc: 0.8469
[geglu] epoch 2/5 | train loss: 0.4682 | test acc: 0.9204
[geglu] epoch 3/5 | train loss: 0.3256 | test acc: 0.9337
[geglu] epoch 4/5 | train loss: 0.2636 | test acc: 0.9496
[geglu] epoch 5/5 | train loss: 0.2264 | test acc: 0.9545

Run: swiglu | params: 105,010
[swiglu] epoch 1/5 | train loss: 1.2938 | test acc: 0.8581
[swiglu] epoch 2/5 | train loss: 0.4832 | test acc: 0.9185
[swiglu] epoch 3/5 | train loss: 0.3330 | test acc: 0.9343
[swiglu] epoch 4/5 | train loss: 0.2629 | test acc: 0.9505
[swiglu] epoch 5/5 | train loss: 0.2165 | test acc: 0.9574
baseline: final 